# Energy and Force Model Showcase

This notebook demonstrates the **EnergyForceModel** wrapper in kgcnn_torch, which
computes forces as the negative gradient of the predicted energy with respect to
atomic positions using PyTorch autograd.

Key concepts:
- Loading the MD17 Revised dataset (aspirin trajectory)
- Wrapping an energy-predicting GNN (SchNet or PAiNN) with `EnergyForceModel`
- Computing forces as F = -dE/dpos via `torch.autograd.grad`
- Scaling energy with `EnergyForceExtensiveLabelScaler`
- Joint energy+force loss with configurable weighting
- Training and evaluating both energy and force predictions
- Running ASE molecular dynamics with the trained model

In [ ]:
import torch
import torch.nn as nn
import numpy as np
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1. Loading the MD17 Revised Dataset

The MD17 Revised dataset contains molecular dynamics trajectories recalculated at
the PBE/def2-SVP level with tight SCF convergence. We load the aspirin trajectory
(100,000 structures) and use a subset of 500 for this demonstration.

We apply graph preprocessors to build range connections and count nodes/edges,
then convert units from kcal/mol to eV.

In [ ]:
from kgcnn_torch.data.datasets.MD17RevisedDataset import MD17RevisedDataset

dataset = MD17RevisedDataset("aspirin")
print("Number of steps:", len(dataset))

# Use 500 samples for demonstration
# The dataset stores PyG Data objects with: z (atomic numbers), pos, energy, force
pyg_list = [dataset[i].clone() for i in range(500)]


def add_radius_edges(data, cutoff=5.0):
    """Add edges between atoms within a cutoff distance (no torch-cluster needed)."""
    pos = data.pos
    diff = pos.unsqueeze(0) - pos.unsqueeze(1)  # (N, N, 3)
    dist = torch.norm(diff, dim=-1)  # (N, N)
    mask = (dist < cutoff) & (dist > 0)  # exclude self-loops
    src, dst = torch.where(mask)
    data.edge_index = torch.stack([src, dst], dim=0)
    return data


for i in range(len(pyg_list)):
    pyg_list[i] = add_radius_edges(pyg_list[i], cutoff=5.0)

print("Sample data keys:", pyg_list[0].keys())
print(f"Num atoms: {pyg_list[0].z.shape[0]}, Num edges: {pyg_list[0].edge_index.shape[1]}")

## 2. Scale Energy Using EnergyForceExtensiveLabelScaler

We use `EnergyForceExtensiveLabelScaler` to remove per-atom energy offsets via
ridge regression on atom counts. Forces are scaled by the same factor as energy
(when `standardize_scale=True`). Here we disable scale normalization for simplicity.

In [ ]:
from kgcnn_torch.data.transform import EnergyForceExtensiveLabelScaler

# Extract energies and forces, convert from kcal/mol to eV
eng = np.array([d.energy.numpy().flatten() for d in pyg_list]) * 0.043  # kcal/mol -> eV
forces_list = [d.force.numpy() * 0.043 for d in pyg_list]
atoms_list = [d.z.numpy() for d in pyg_list]

# Fit scaler - removes per-atom energy offset via ridge regression
scaler = EnergyForceExtensiveLabelScaler(standardize_scale=False)
scaled_energy, scaled_forces = scaler.fit_transform(
    y=eng, force=forces_list, atomic_number=atoms_list
)

print(f"Original energy range: [{eng.min():.3f}, {eng.max():.3f}] eV")
print(f"Scaled energy range: [{scaled_energy.min():.3f}, {scaled_energy.max():.3f}]")

# Update PyG data with scaled values
for i in range(len(pyg_list)):
    pyg_list[i].energy = torch.tensor(scaled_energy[i], dtype=torch.float)
    pyg_list[i].force = torch.tensor(scaled_forces[i], dtype=torch.float)
    pyg_list[i].y = pyg_list[i].energy  # Set y for the loss function

# Store atomic numbers for later use
atomic_numbers = atoms_list

print(f"Generated {len(pyg_list)} PyG Data objects")
print(f"Sample: energy={pyg_list[0].y}, force shape={pyg_list[0].force.shape}")

## 3. The EnergyForceModel Wrapper

The `EnergyForceModel` wraps any energy-predicting GNN and computes forces
as the negative gradient of the energy with respect to atomic positions:

$$\vec{F}_i = -\frac{\partial E_{\text{total}}}{\partial \vec{r}_i}$$

This requires `data.pos.requires_grad_(True)` so that PyTorch can compute
the gradient via `torch.autograd.grad`.

In [ ]:
from kgcnn_torch.models.schnet import SchNetModel
from kgcnn_torch.models.force import EnergyForceModel

# Build the energy model (SchNet) - matching Keras config exactly:
# Keras uses last_mlp: [128, 64, 1] with use_output_mlp=False,
# meaning the per-node MLP reduces to 1 dim BEFORE graph pooling.
energy_model = SchNetModel(
    node_dim=64,
    depth=4,
    units=128,
    gauss_bins=20,
    gauss_distance=4.0,
    gauss_sigma=0.4,
    gauss_offset=0.0,
    interaction_activation="shifted_softplus",
    interaction_pooling="sum",
    node_pooling="sum",
    last_mlp_units=[128, 64, 1],
    last_mlp_activation=["shifted_softplus", "shifted_softplus", "linear"],
    num_targets=1,
    output_embedding="graph",
    use_node_embedding=True,
    num_embeddings=95,
    make_distance=True,
    expand_distance=True,
    use_output_mlp=False,
)

# Wrap with EnergyForceModel
model = EnergyForceModel(
    energy_model=energy_model,
    coordinate_input="pos",
    output_as_dict=True,            # Returns {"energy": ..., "force": ...}
    output_squeeze_states=True,
    is_physical_force=True,         # force = -gradient (physical convention)
)

print("EnergyForceModel configuration:")
print(model.get_config())
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

## 4. Test Forward Pass

Let us verify the model produces both energy and force predictions.

In [ ]:
from torch_geometric.loader import DataLoader

# Test with a small batch
test_batch_loader = DataLoader(pyg_list[:4], batch_size=4, shuffle=False)
test_batch = next(iter(test_batch_loader))

# Positions must require gradients for autograd force computation
test_batch.pos.requires_grad_(True)

model.eval()
with torch.set_grad_enabled(True):
    output = model(test_batch)

print("Output keys:", output.keys())
print("Energy shape:", output["energy"].shape)
print("Force shape:", output["force"].shape)
print("Energy values:", output["energy"].detach().numpy().flatten())

## 5. Energy+Force Loss Function

We define a combined loss that weights energy and force contributions:

$$\mathcal{L} = w_E \cdot \text{MAE}(E_{\text{pred}}, E_{\text{true}}) + w_F \cdot \text{MAE}(F_{\text{pred}}, F_{\text{true}})$$

The force weight is typically much larger (e.g., 20x) because there are more
force components (3 * N_atoms) than energy values (1 per molecule).

In [ ]:
class EnergyForceLoss(nn.Module):
    """Combined energy and force loss with configurable weights."""

    def __init__(self, energy_weight=1.0, force_weight=20.0):
        super().__init__()
        self.energy_weight = energy_weight
        self.force_weight = force_weight
        self.mae = nn.L1Loss()

    def forward(self, pred_dict, batch):
        """Compute combined loss.

        Args:
            pred_dict: Dict with 'energy' and 'force' predictions.
            batch: PyG Batch object with .y (energy) and .force targets.
        """
        # Energy loss
        energy_pred = pred_dict["energy"]
        energy_target = batch.y
        if energy_target.dim() == 1:
            energy_target = energy_target.unsqueeze(-1)
        loss_energy = self.mae(energy_pred, energy_target)

        # Force loss
        force_pred = pred_dict["force"]
        force_target = batch.force
        loss_force = self.mae(force_pred, force_target)

        return self.energy_weight * loss_energy + self.force_weight * loss_force

loss_fn = EnergyForceLoss(energy_weight=1.0, force_weight=20.0)
print("Loss function created with weights: energy={}, force={}".format(
    loss_fn.energy_weight, loss_fn.force_weight))

## 6. Training with Energy+Force Supervision

We use the `fit()` function from `kgcnn_torch.training.trainer`. The loss function
receives the dict output from EnergyForceModel and the batch object (which contains
both energy and force targets).

In [ ]:
from kgcnn_torch.training.trainer import fit

# Split data
n_train = int(0.8 * len(pyg_list))
train_data = pyg_list[:n_train]
test_data = pyg_list[n_train:]

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

# Create model and optimizer (matching Keras architecture exactly)
energy_model = SchNetModel(
    node_dim=64,
    depth=4,
    units=128,
    gauss_bins=20,
    gauss_distance=4.0,
    gauss_sigma=0.4,
    gauss_offset=0.0,
    interaction_activation="shifted_softplus",
    interaction_pooling="sum",
    node_pooling="sum",
    last_mlp_units=[128, 64, 1],
    last_mlp_activation=["shifted_softplus", "shifted_softplus", "linear"],
    num_targets=1,
    output_embedding="graph",
    use_node_embedding=True,
    num_embeddings=95,
    make_distance=True,
    expand_distance=True,
    use_output_mlp=False,
)
model = EnergyForceModel(
    energy_model=energy_model,
    coordinate_input="pos",
    output_as_dict=True,
    output_squeeze_states=True,
)
model = model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)
loss_fn = EnergyForceLoss(energy_weight=1.0, force_weight=20.0)

# Train (Keras uses 1500 epochs)
history = fit(
    model=model,
    train_loader=train_loader,
    val_loader=test_loader,
    optimizer=optimizer,
    loss_fn=loss_fn,
    epochs=1500,
    device=device,
    verbose=1,
)

print(f"\nFinal train loss: {history['train_loss'][-1]:.4f}")
print(f"Final val loss: {history['val_loss'][-1]:.4f}")

## 7. Plot Training Curves

In [ ]:
from kgcnn_torch.utils.plots import plot_train_test_loss

plot_train_test_loss(
    [history],
    loss_name="train_loss",
    val_loss_name="val_loss",
    model_name="EnergyForce-SchNet",
    data_unit="eV",
    dataset_name="MD17Revised (aspirin)",
);

## 8. Evaluate Energy and Force Predictions

We evaluate the model on the test set and compute separate MAE for energy and forces.

In [ ]:
model.eval()
all_energy_pred, all_energy_true = [], []
all_force_pred, all_force_true = [], []

for batch in test_loader:
    batch = batch.to(device)
    batch.pos.requires_grad_(True)
    output = model(batch)

    all_energy_pred.append(output["energy"].detach().cpu().numpy())
    all_energy_true.append(batch.y.detach().cpu().numpy())
    all_force_pred.append(output["force"].detach().cpu().numpy())
    all_force_true.append(batch.force.detach().cpu().numpy())

energy_pred = np.concatenate(all_energy_pred).flatten()
energy_true = np.concatenate(all_energy_true).flatten()
force_pred = np.concatenate(all_force_pred)
force_true = np.concatenate(all_force_true)

energy_mae = np.mean(np.abs(energy_pred - energy_true))
force_mae = np.mean(np.abs(force_pred - force_true))

print(f"Energy MAE: {energy_mae:.4f}")
print(f"Force MAE:  {force_mae:.4f}")
print(f"Force RMSE: {np.sqrt(np.mean((force_pred - force_true)**2)):.4f}")

## 9. Scatter Plots: Predicted vs True

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Energy
ax1.scatter(energy_true, energy_pred, alpha=0.4, s=10)
lims = [min(energy_true.min(), energy_pred.min()),
        max(energy_true.max(), energy_pred.max())]
ax1.plot(lims, lims, 'r-', linewidth=1)
ax1.set_xlabel("True Energy")
ax1.set_ylabel("Predicted Energy")
ax1.set_title(f"Energy (MAE={energy_mae:.4f})")

# Forces (flatten all components)
ax2.scatter(force_true.flatten(), force_pred.flatten(), alpha=0.1, s=5)
lims = [min(force_true.min(), force_pred.min()),
        max(force_true.max(), force_pred.max())]
ax2.plot(lims, lims, 'r-', linewidth=1)
ax2.set_xlabel("True Force")
ax2.set_ylabel("Predicted Force")
ax2.set_title(f"Forces (MAE={force_mae:.4f})")

plt.tight_layout()
plt.show()

## 10. Using PAiNN as the Energy Model

PAiNN is an equivariant GNN that maintains both scalar and vector features.
It can be used as a drop-in replacement for SchNet inside EnergyForceModel.

In [ ]:
from kgcnn_torch.models.painn import PAiNNModel

painn_energy_model = PAiNNModel(
    node_dim=128,
    depth=3,
    units=128,
    num_radial=20,
    cutoff=5.0,
    conv_activation="swish",
    update_activation="swish",
    node_pooling="sum",
    output_units=[128],
    output_activation="swish",
    num_targets=1,
    output_embedding="graph",
    use_node_embedding=True,
    num_embeddings=95,
)

painn_force_model = EnergyForceModel(
    energy_model=painn_energy_model,
    coordinate_input="pos",
    output_as_dict=True,
    output_squeeze_states=True,
)

print(f"PAiNN EnergyForceModel parameters: {sum(p.numel() for p in painn_force_model.parameters()):,}")
print("Model can be trained with the same EnergyForceLoss and fit() function.")

## 11. MolDynamicsModelPredictor for ASE Integration

The `MolDynamicsModelPredictor` wraps the trained model with pre- and postprocessors
to make it compatible with molecular dynamics programs like ASE. It handles:

- **Preprocessing**: Rebuilding range connections from atomic coordinates
- **Model inference**: Running the trained EnergyForceModel on PyG data
- **Postprocessing**: Inverse-scaling predictions back to physical units

Since the PyTorch model uses PyG batching internally, we create a small adapter
(`PyGModelAdapter`) that bridges the `MemoryGraphList` tensor format used by the
dynamics predictor with the PyG `Batch` format expected by the model.

In [ ]:
from kgcnn_torch.molecule.dynamics.base import MolDynamicsModelPredictor
from kgcnn_torch.graph.postprocessor import ExtensiveEnergyForceScalerPostprocessor
from kgcnn_torch.graph.preprocessor import SetRange, CountNodesAndEdges
from kgcnn_torch.data.base import MemoryGraphList
from kgcnn_torch.graph.base import GraphDict
from torch_geometric.data import Data, Batch


class PyGModelAdapter:
    """Adapts a PyG-based EnergyForceModel for use with MolDynamicsModelPredictor.

    Converts padded tensor inputs (from MemoryGraphList.tensor()) to a PyG Batch,
    runs the model, and returns padded output tensors.
    """

    def __init__(self, model):
        self.model = model

    def __call__(self, tensor_input, **kwargs):
        z_padded, pos_padded, edge_idx_padded, total_nodes, total_edges = tensor_input
        batch_size = z_padded.shape[0]
        data_list = []

        for b in range(batch_size):
            n = int(total_nodes[b])
            m = int(total_edges[b])

            z = torch.tensor(z_padded[b, :n], dtype=torch.long)
            pos = torch.tensor(pos_padded[b, :n], dtype=torch.float32)
            # KGCNN convention (target, source) -> PyG (source, target)
            ei_raw = edge_idx_padded[b, :m]
            edge_index = torch.tensor(ei_raw[:, [1, 0]].T, dtype=torch.long)

            data_list.append(Data(z=z, pos=pos, edge_index=edge_index, num_nodes=n))

        pyg_batch = Batch.from_data_list(data_list)
        pyg_batch.pos.requires_grad_(True)

        self.model.eval()
        with torch.set_grad_enabled(True):
            output = self.model(pyg_batch)

        # Split forces back to padded format (for MolDynamicsModelPredictor indexing)
        energy = output["energy"].detach()
        force = output["force"].detach()

        max_atoms = pos_padded.shape[1]
        padded_forces = torch.zeros(batch_size, max_atoms, 3)
        batch_indices = pyg_batch.batch
        for b in range(batch_size):
            mask = batch_indices == b
            n = mask.sum().item()
            padded_forces[b, :n] = force[mask]

        return {"energy": energy, "forces": padded_forces}


# Move model to CPU for ASE compatibility
model_cpu = model.cpu()

# Model input specification (matches the graph preprocessor output)
model_inputs = [
    {"name": "atomic_number", "dtype": "int32", "shape": [None]},
    {"name": "node_coordinates", "dtype": "float32", "shape": [None, 3]},
    {"name": "range_indices", "dtype": "int64", "shape": [None, 2]},
    {"name": "total_nodes", "dtype": "int64", "shape": ()},
    {"name": "total_ranges", "dtype": "int64", "shape": ()},
]

dyn_model = MolDynamicsModelPredictor(
    model=PyGModelAdapter(model_cpu),
    model_inputs=model_inputs,
    model_outputs={"energy": "energy", "forces": "forces"},
    graph_preprocessors=[
        SetRange(node_coordinates="node_coordinates", overwrite=False),
        CountNodesAndEdges(
            total_edges="total_ranges", count_edges="range_indices",
            count_nodes="atomic_number", total_nodes="total_nodes"
        )
    ],
    graph_postprocessors=[
        ExtensiveEnergyForceScalerPostprocessor(
            scaler, energy="energy", force="forces", atomic_number="atomic_number"
        )
    ],
    store_last_input=True,
    update_from_last_input=["range_indices"],
    update_from_last_input_skip=3,
    use_predict=False,
)

# Test the predictor on a single sample (build MemoryGraphList from PyG Data)
sample = dataset[0]
test_graph = GraphDict({
    "atomic_number": sample.z.numpy(),
    "node_coordinates": sample.pos.numpy(),
})
test_mgl = MemoryGraphList([test_graph])
test_out = dyn_model(test_mgl)
print("Predictor output keys:", test_out[0].keys())
print("Energy:", test_out[0]["energy"])
print("Forces shape:", test_out[0]["forces"].shape)

## 12. ASE Calculator and Molecular Dynamics

We use `KgcnnSingleCalculator` to make the trained model available as an ASE
calculator. Combined with `AtomsToGraphConverter`, this bridges the ASE `Atoms`
object to the kgcnn graph representation.

We then run a short Velocity Verlet MD simulation at 300K.

In [ ]:
from ase import Atoms, units
from kgcnn_torch.molecule.dynamics.ase_calc import KgcnnSingleCalculator, AtomsToGraphConverter

# Create ASE Atoms from the first structure in the dataset
sample = dataset[0]
atoms = Atoms(sample.z.numpy(), positions=sample.pos.numpy())
print("ASE Atoms:", atoms)

# Create converter: maps ASE Atoms properties to graph dict properties
conv = AtomsToGraphConverter({
    "atomic_number": "get_atomic_numbers",
    "node_coordinates": "get_positions"
})

# Create ASE calculator
calc = KgcnnSingleCalculator(model_predictor=dyn_model, atoms_converter=conv)
atoms.calc = calc

# Test the calculator
calc.calculate(atoms)
print("\nCalculator results:")
print(f"  Energy: {calc.results['energy']:.3f} eV")
print(f"  Forces shape: {calc.results['forces'].shape}")
print(f"  Max force: {np.abs(calc.results['forces']).max():.3f} eV/Ang")

In [ ]:
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution
from ase.md.verlet import VelocityVerlet

# Set the momenta corresponding to T=300K
MaxwellBoltzmannDistribution(atoms, temperature_K=300)

# Run MD with constant energy using the Velocity Verlet algorithm
dyn = VelocityVerlet(atoms, 1 * units.fs)  # 1 fs time step


def printenergy(a):
    """Print the potential, kinetic and total energy per atom."""
    epot = a.get_potential_energy() / len(a)
    ekin = a.get_kinetic_energy() / len(a)
    print('Energy per atom: Epot = %.3feV  Ekin = %.3feV (T=%3.0fK)  '
          'Etot = %.3feV' % (epot, ekin, ekin / (1.5 * units.kB), epot + ekin))


# Run the dynamics
printenergy(atoms)
for i in range(20):
    dyn.run(10)
    printenergy(atoms)

## Summary

In this notebook we demonstrated:

1. **MD17 Revised Dataset**: Loading real molecular dynamics data (aspirin) with proper unit conversion
2. **EnergyForceExtensiveLabelScaler**: Removing per-atom energy offsets via ridge regression
3. **EnergyForceModel**: Wrapping any energy-predicting GNN and computing forces via autograd
4. **EnergyForceLoss**: Combined loss with energy/force weighting
5. **Training**: Joint energy+force supervision using the standard `fit()` trainer
6. **Evaluation**: Separate MAE metrics for energy and force components with scatter plots
7. **Model flexibility**: Both SchNet and PAiNN can serve as the energy backbone
8. **MolDynamicsModelPredictor**: Wrapping the trained model with pre/postprocessors
9. **ASE Integration**: Using `KgcnnSingleCalculator` for molecular dynamics simulation
10. **Velocity Verlet MD**: Running NVE dynamics at 300K with energy conservation